# Medical Cost Personal - Insurance Charges Prediction

**Problem type:** Regression

**Goal:** Predict the **yearly medical charges** of an insured person from age, BMI, smoking, health conditions, etc.

**Target (dependent variable):** `annual_medical_charges` (USD per year)

**Data:** synthetic (fake) dataset created by `generate_data.py` in this folder -> `data/medical_insurance_members.csv`
(about 100,000 rows and 50+ columns). The data has **problems added on purpose** (codes, numbers saved as text, missing values,
outliers, wrong values, dates as text, messy text, useless / duplicate columns, duplicate rows) so we can practice the full cleaning process.

## Notebook plan
- **A. Data Cleaning** - drop useless columns, rename, data types, missing values, outliers, duplicates, one-value columns
- **B. EDA** - target distribution, correlation, feature charts
- **C. Feature Engineering** - binning, encoding, data splitting (80/15/5), scaling, feature selection
- **D. Feature Extraction** - PCA, Factor Analysis, Cluster Analysis
- **E. Machine Learning** - 20 models, cross validation, train/test evaluation, grid search tuning, compare, save, load, predict on the 5% unseen data

## 0. Import libraries

In [ ]:
# If a library is missing, run this in a cell:
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.cluster import KMeans
from sklearn.base import clone
from sklearn.metrics import silhouette_score

# ---- Regression models ----
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, ARDRegression,
                                  HuberRegressor, SGDRegressor)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor,
                              AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.svm import SVR, LinearSVR
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor

# ---- Evaluation metrics ----
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error,
                             median_absolute_error, explained_variance_score, max_error)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)   # show numbers with 4 decimals, not 1e+03
sns.set_style("whitegrid")

RANDOM_STATE = 123

## 1. Load the data

If the file `data/medical_insurance_members.csv` does not exist, run `python generate_data.py` first.

In [ ]:
df = pd.read_csv("data/medical_insurance_members.csv")

print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

# A. Data Cleaning

## A0. Drop columns that are not useful

`member_id`, `full_name`, `email`, `phone_number` only identify the person, and `date_of_birth` is not needed because we have `age`.

In [ ]:
drop_columns = [
    "Member ID", "Full Name", "E-mail", "Phone #",
    "DOB",
]
df = df.drop(columns=drop_columns)
print("Shape after dropping:", df.shape)

## A1. Rename columns

The column names are a mix of styles (`Annual Income ($)`, `CreditScore`, `No. of Dependents`). Short, lower-case names with `_` are easier to use in code.
1. A general rule for all columns: lower case, and every space / symbol becomes `_`
2. Some names are still not clear (for example `creditscore`), so we rename them one by one

In [ ]:
print("Before:", df.columns.tolist())

# Step 1: lower case + every space / symbol becomes "_"   ("Annual Income ($)" -> "annual_income")
df.columns = (df.columns.str.strip()
              .str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True)
              .str.strip("_"))

# Step 2: rename the names that are still not clear
rename_map = {
    "age_years": "age",
    "children": "num_children",
    "bp_systolic": "blood_pressure_systolic",
    "bp_diastolic": "blood_pressure_diastolic",
    "charges": "annual_medical_charges",
}
df = df.rename(columns=rename_map)

TARGET = "annual_medical_charges"
print("\nAfter:", df.columns.tolist())

## A2. Data types and wrong values

Let's first see the data types. Many columns that should be numbers are `object` (text).

In [ ]:
print(df.dtypes.value_counts())
df.dtypes

### A2.1 Numbers saved as text

These columns contain numbers written as text: money (`$1,250.00`), numbers with commas (`12,500`), percent (`12.5%`) or numbers with a unit (`34 yrs`, `12.5 km`). Some missing values are written as text too (`N/A`, `unknown`, `?`, `-`).

We keep only the digits, the `.` and the `-`, then `pd.to_numeric(errors="coerce")` changes the text to numbers (anything that is not a number becomes `NaN`).

In [ ]:
text_number_cols = [
    "age", "weight_kg", "height_cm",
    "annual_income", "deductible", "coverage_level",
    "annual_medical_charges",
]

for col in text_number_cols:
    print(f"{col:30s} examples: {list(df[col].dropna().unique()[:5])}")
    df[col] = df[col].astype(str).str.replace(r"[^0-9.\-]", "", regex=True)   # keep only digits, "." and "-"
    df[col] = pd.to_numeric(df[col], errors="coerce")                         # text -> number (empty -> NaN)

df[text_number_cols].describe().T

### A2.2 Text written in different ways

The same value is written with different upper / lower case and extra spaces (`new york`, `NEW YORK`, `  New York`). Python sees them as **different** values, so we remove the spaces and make everything lower case.

In [ ]:
text_cols = [
    "city", "exercise_frequency", "diet_quality",
]

print("Before:", list(df[text_cols[0]].dropna().unique()[:8]))
for col in text_cols:
    df[col] = df[col].str.strip().str.lower()
print("After :", list(df[text_cols[0]].dropna().unique()[:8]))

### A2.3 Short codes -> full values

Some columns use short codes (`M` / `F`, `Y` / `N`, `T` / `F`, `FT`, `HS`, ...). We change them to clear values (`male` / `female`, `yes` / `no`, ...) so the data is easy to read.

In [ ]:
value_maps = {
    "gender": {"M": "male", "F": "female"},
    "marital_status": {"S": "single", "M": "married", "D": "divorced", "W": "widowed"},
    "education": {"HS": "high_school", "BSC": "bachelor", "MSC": "master", "PHD": "phd"},
    "marketing_opt_in": {"Y": "yes", "N": "no"},
    "smoker": {"Y": "yes", "N": "no"},
    "alcohol_use": {"N": "none", "M": "moderate", "H": "heavy"},
    "occupation_risk": {"L": "low", "M": "medium", "H": "high"},
    "family_history_heart": {"T": "yes", "F": "no"},
    "has_diabetes": {"Y": "yes", "N": "no"},
}

for col, mapping in value_maps.items():
    df[col] = df[col].replace(mapping)
    print(f"{col:25s} -> {list(df[col].dropna().unique()[:6])}")

### A2.4 Dates saved as text -> datetime

Dates are saved as text, and not always in the same format (`2025-03-15`, `15/03/2025`, `03-15-2025`). We change them to `datetime` using the correct format. A model can not use a date directly, so we create **numbers** from it (how many days ago, month, hour, ...) and then drop the original date column.

The data was exported on **2025-12-31**, so we count the days until that date.

In [ ]:
REFERENCE_DATE = pd.Timestamp("2025-12-31")   # the day the data was exported

date_formats = {
    "policy_start_date": "%Y-%m-%d",
}
for col, fmt in date_formats.items():
    print(f"{col:25s} examples: {list(df[col].dropna().unique()[:3])}")
    df[col] = pd.to_datetime(df[col], format=fmt)

# New numeric features from the dates
df["policy_start_days_ago"] = (REFERENCE_DATE - df["policy_start_date"]).dt.days

# The original date columns are not needed anymore
df = df.drop(columns=list(date_formats))

### A2.5 Categories saved as numbers

`region_code` looks like a number (1, 2, 3, ...) but it is a **category** (region 3 is not "3 times" region 1). We change it to text so it will be one-hot encoded later.

In [ ]:
category_cols = [
    "region_code",
]
for col in category_cols:
    df[col] = df[col].astype(str)

In [ ]:
print("After changing the data types:")
print(df.dtypes.value_counts())
df.dtypes

## A3. Missing values

### Percentage of missing values in each column

(This now includes the hidden missing values like `N/A`, `unknown`, `?` that became `NaN` in step A2.1.)

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().sum() / len(df) * 100).round(2),
})
missing = missing[missing["Missing Count"] > 0].sort_values("Missing %", ascending=False)
print(missing)

missing["Missing %"].plot(kind="bar", figsize=(12, 4), title="Missing values (%)")
plt.axhline(50, color="red", linestyle="--", label="50%")
plt.ylabel("%")
plt.legend()
plt.show()

### Handle the missing values

1. Columns with **more than 50% missing** have too little information -> **drop the column**.
2. The other columns have a small % missing -> **fill** them: numeric columns with the **mean**, text columns with the **mode** (most frequent value).
   (Removing all rows with a missing value would delete a big part of the data, because the missing values are spread over many columns.)

In [ ]:
# 1) drop columns with more than 50% missing
high_missing_cols = missing[missing["Missing %"] > 50].index.tolist()
print("Columns with more than 50% missing (dropped):", high_missing_cols)
df = df.drop(columns=high_missing_cols)

# 2) fill the rest: numeric -> mean, text -> mode
for col in df.columns:
    if df[col].isna().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].mean())
            print(f"{col:30s} filled with the mean")
        else:
            df[col] = df[col].fillna(df[col].mode()[0])
            print(f"{col:30s} filled with the mode")

print("\nMissing values after handling:", df.isna().sum().sum())

## A4. Outliers and wrong values

### Visualize the outliers (box plots)

A point outside the box *whiskers* is an outlier by the **IQR rule**: lower than `Q1 - 1.5*IQR` or higher than `Q3 + 1.5*IQR` (IQR = Q3 - Q1).

In [ ]:
outlier_cols = [
    "age", "bmi", "blood_pressure_systolic",
    "steps_per_day", "weight_kg", "cholesterol_mg_dl",
    "glucose_mg_dl", "annual_income", "sleep_hours",
]

n_rows = (len(outlier_cols) + 3) // 4
fig, axes = plt.subplots(n_rows, 4, figsize=(16, 3.5 * n_rows))
axes = axes.flatten()
for i, col in enumerate(outlier_cols):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(col)
for j in range(len(outlier_cols), len(axes)):
    axes[j].axis("off")   # hide empty charts
plt.tight_layout()
plt.show()

In [ ]:
# Count the outliers in each column (IQR rule)
outlier_table = []
for col in outlier_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_table.append({"Column": col, "Min": df[col].min(), "Max": df[col].max(), "Lower limit": lower, "Upper limit": upper,
                          "Outliers": n_outliers, "Outliers %": round(n_outliers / len(df) * 100, 2)})

pd.DataFrame(outlier_table)

### Handle the outliers

1. **Wrong / impossible values** (like age = 150, a negative mileage, a credit score of 9999) are errors, not real values -> we replace them with the **median** (imputation). We use a *valid range* for each column.
2. **Extreme but possible values** -> we **cap** them: values above the upper IQR limit become the upper limit (and the same for the lower limit). We keep all the rows.

We do **not** change the target.

In [ ]:
# 1) Wrong / impossible values -> replace them with the median of the correct values
valid_ranges = {
    "age": (18, 100),
    "bmi": (12, 70),
    "blood_pressure_systolic": (70, 250),
    "steps_per_day": (0, 50000),
    "weight_kg": (30, 300),
}
for col, (low, high) in valid_ranges.items():
    wrong = (df[col] < low) | (df[col] > high)
    median = df.loc[~wrong, col].median()
    df[col] = df[col].mask(wrong, median)
    print(f"{col:30s} {wrong.sum():5d} wrong values -> replaced with the median ({median:.2f})")

# 2) Extreme values -> cap them with the IQR limits
iqr_cap_cols = [
    "cholesterol_mg_dl", "glucose_mg_dl", "annual_income",
    "sleep_hours",
]
for col in iqr_cap_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

df[outlier_cols].describe().T

In [ ]:
# Box plots after handling the outliers
n_rows = (len(outlier_cols) + 3) // 4
fig, axes = plt.subplots(n_rows, 4, figsize=(16, 3.5 * n_rows))
axes = axes.flatten()
for i, col in enumerate(outlier_cols):
    sns.boxplot(x=df[col], ax=axes[i], color="lightgreen")
    axes[i].set_title(col + " (after)")
for j in range(len(outlier_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

### Whole numbers back to `int`

These columns are whole numbers (ages, counts, scores). They became `float` because of the missing values and the median / mean filling, so we round them and change them back to `int`.

In [ ]:
int_cols = [
    "age",
]
for col in int_cols:
    df[col] = df[col].round().astype(int)
df[int_cols].dtypes

## A5. Duplicate rows and duplicate columns

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicate rows:", df.shape)

In [ ]:
# Duplicate columns = two columns with exactly the same values
duplicate_columns = []
columns = df.columns.tolist()
for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        if df[columns[i]].equals(df[columns[j]]):
            print(f"'{columns[j]}' is the same as '{columns[i]}'")
            duplicate_columns.append(columns[j])

df = df.drop(columns=duplicate_columns)
print("Duplicate columns dropped:", duplicate_columns)

## A6. Columns with only one value

A column with one value (or almost only one value, more than 99% the same) gives no information to the model.

In [ ]:
one_value_cols = []
for col in df.columns:
    if col == TARGET:
        continue
    top_value_percent = df[col].value_counts(normalize=True).iloc[0] * 100
    if df[col].nunique() <= 1 or top_value_percent > 99:
        one_value_cols.append(col)
        print(f"{col:25s} {df[col].nunique()} unique value(s), most frequent value = {top_value_percent:.2f}% of rows")

df = df.drop(columns=one_value_cols)
print("\nColumns dropped:", one_value_cols)
print("Shape after cleaning:", df.shape)

In [ ]:
df.info()

# B. Exploratory Data Analysis (EDA)

## B1. Target (dependent variable): distribution

In [ ]:
print(df[TARGET].describe())
print("Skewness:", round(df[TARGET].skew(), 2))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(df[TARGET], kde=True, ax=axes[0])
axes[0].set_title(f"{TARGET} - histogram")
sns.boxplot(x=df[TARGET], ax=axes[1])
axes[1].set_title(f"{TARGET} - box plot")
sns.histplot(np.log1p(df[TARGET]), kde=True, ax=axes[2], color="orange")
axes[2].set_title(f"log({TARGET}) - histogram")
plt.show()

Skewness > 1 means the target has a long right tail (a few very large values). The log chart shows how it would look after a log transform (an idea you can try to improve the models).

## B2. Correlation chart

In [ ]:
numeric_df = df.select_dtypes(include="number")
corr = numeric_df.corr()

size = max(8, len(corr) * 0.5)
plt.figure(figsize=(size, size * 0.8))
sns.heatmap(corr, annot=len(corr) <= 15, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix")
plt.show()

In [ ]:
# Correlation of each feature with the target
target_corr = corr[TARGET].drop(TARGET).sort_values()
target_corr.plot(kind="barh", figsize=(8, max(4, len(target_corr) * 0.3)), color="teal")
plt.title(f"Correlation with {TARGET}")
plt.show()

## B3. Independent features (charts)

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print(len(num_cols), "numeric features:", num_cols)
print(len(cat_cols), "categorical features:", cat_cols)

### Numeric features - histograms

In [ ]:
n_rows = (len(num_cols) + 3) // 4
df[num_cols].hist(bins=30, figsize=(16, 3 * n_rows), layout=(n_rows, 4), color="steelblue", edgecolor="black")
plt.tight_layout()
plt.show()

### Categorical features - count plots

In [ ]:
n_rows = (len(cat_cols) + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    sns.countplot(x=df[col], order=df[col].value_counts().index, ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis="x", rotation=45)
for j in range(len(cat_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

### Numeric features vs target (scatter plots on a random sample of 5,000 rows, so the charts are fast and readable)

In [ ]:
sample = df.sample(n=5000, random_state=RANDOM_STATE)
n_rows = (len(num_cols) + 3) // 4
fig, axes = plt.subplots(n_rows, 4, figsize=(16, 3.5 * n_rows))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].scatter(sample[col], sample[TARGET], alpha=0.3, s=8)
    axes[i].set_title(col)
for j in range(len(num_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

### Categorical features vs target (box plots)

In [ ]:
n_rows = (len(cat_cols) + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 4.5 * n_rows))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    sns.boxplot(x=df[col], y=df[TARGET], ax=axes[i], showfliers=False)
    axes[i].set_title(f"{TARGET} by {col}")
    axes[i].tick_params(axis="x", rotation=45)
for j in range(len(cat_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

# C. Feature Engineering

## C1. Feature binning (and new features)

**Binning** = putting a numeric column into groups (for example age 18-25, 26-35, ...).

We create `age_group` and `bmi_category` (the official BMI groups).

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[17, 30, 45, 60, 100], labels=["18-30", "31-45", "46-60", "60+"])
df["bmi_category"] = pd.cut(df["bmi"], bins=[0, 18.5, 25, 30, 100], labels=["underweight", "normal", "overweight", "obese"])

print(df["age_group"].value_counts())
print(df["bmi_category"].value_counts())

## C2. Feature encoding

Models need numbers, not text.
- Columns with **2 values** -> **Label Encoding** (0 / 1)
- Columns with **more than 2 values** -> **One-Hot Encoding** (one new 0/1 column for each value, `drop_first=True` removes one to avoid repeating information)

In [ ]:
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
multi_cols = [col for col in cat_cols if df[col].nunique() > 2]
print("Label Encoding  :", binary_cols)
print("One-Hot Encoding:", multi_cols)

# Label Encoding
for col in binary_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

# One-Hot Encoding
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, dtype=int)

# Clean the column names (XGBoost does not accept some characters like < [ ] in names)
df.columns = (df.columns.str.replace(" ", "_").str.replace("<", "less_")
              .str.replace("[", "").str.replace("]", "").str.replace(",", ""))

print("\nShape after encoding:", df.shape)
df.head()

## C3. Data splitting (80% train / 15% test / 5% unseen)

We split **before scaling and feature selection**. The scaler and the selector must learn only from the training data - if they see the test data, the results will look better than reality (this is called **data leakage**).

- **Train (80%)** - to train the models
- **Test (15%)** - to evaluate and compare the models
- **Unseen (5%)** - kept aside until the end; used only once to check the final saved model

`random_state = 123` so we get the same split every time.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Step 1: 80% train, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

# Step 2: split the 20% into 15% test and 5% unseen (0.25 x 20% = 5%)
X_test, X_unseen, y_test, y_unseen = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE)

print("Train :", X_train.shape, f"{len(X_train) / len(X) * 100:.1f}%")
print("Test  :", X_test.shape, f"{len(X_test) / len(X) * 100:.1f}%")
print("Unseen:", X_unseen.shape, f"{len(X_unseen) / len(X) * 100:.1f}%")

## C4. Feature scaling

`StandardScaler` changes every feature to mean = 0 and standard deviation = 1. Models like KNN, SVM, Logistic Regression and Neural Networks need this. We **fit** the scaler on the training data only, then **transform** test and unseen data.

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
X_unseen_scaled = pd.DataFrame(scaler.transform(X_unseen), columns=X.columns, index=X_unseen.index)

X_train_scaled.describe().T[["mean", "std"]].round(2).head(10)

## C5. Feature selection

After encoding we have many features. `SelectKBest` gives every feature a score (`f_regression` = statistical test between the feature and the target) and keeps the best **K** features. We also show the Random Forest feature importance as a second opinion.

In [ ]:
K = min(20, X_train_scaled.shape[1])   # number of features to keep (you can change it)

selector = SelectKBest(score_func=f_regression, k=K)
selector.fit(X_train_scaled, y_train)

feature_scores = pd.DataFrame({"Feature": X.columns, "Score": selector.scores_})
feature_scores = feature_scores.sort_values("Score", ascending=False)

feature_scores.head(30).plot(kind="barh", x="Feature", y="Score", figsize=(8, 9), legend=False)
plt.gca().invert_yaxis()
plt.title("SelectKBest scores (top 30)")
plt.show()

selected_features = X.columns[selector.get_support()].tolist()
print(f"Selected {len(selected_features)} features:", selected_features)

In [ ]:
# Second opinion: Random Forest feature importance (on 20,000 training rows to be fast)
rf_rows = X_train_scaled.sample(n=20000, random_state=RANDOM_STATE)
rf_check = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_check.fit(rf_rows, y_train.loc[rf_rows.index])
importance = pd.Series(rf_check.feature_importances_, index=X.columns).sort_values(ascending=False)

importance.head(20).plot(kind="barh", figsize=(8, 7), color="darkorange")
plt.gca().invert_yaxis()
plt.title("Random Forest feature importance (top 20)")
plt.show()

In [ ]:
# Keep only the selected features
X_train_sel = X_train_scaled[selected_features]
X_test_sel = X_test_scaled[selected_features]
X_unseen_sel = X_unseen_scaled[selected_features]
print(X_train_sel.shape, X_test_sel.shape, X_unseen_sel.shape)

## C6. A smaller training sample for comparing the models

The training data has about 80,000 rows. Training **20 models + cross validation + grid search** on all of them would take hours (especially SVM and KNN). So we take a random sample of **20,000 training rows** to compare and tune the models. At the end, the best model is **re-trained on the full training data**.

In [ ]:
SAMPLE_SIZE = 20000
X_train_small, _, y_train_small, _ = train_test_split(X_train_sel, y_train, train_size=SAMPLE_SIZE,
                                                      random_state=RANDOM_STATE)
print("Training sample:", X_train_small.shape)

# D. Feature Extraction

Feature extraction creates **new features** from the existing ones. Here we use it to **understand** the data (on the 20,000-row training sample); the models in part E use the selected features from C5.

## D1. PCA (Principal Component Analysis)

PCA makes new features (components) that keep as much information (variance) as possible with fewer columns.

In [ ]:
pca = PCA(random_state=RANDOM_STATE)
pca.fit(X_train_small)
cumulative = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
plt.axhline(0.95, color="red", linestyle="--", label="95%")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA - explained variance")
plt.legend()
plt.show()

n_95 = np.argmax(cumulative >= 0.95) + 1
print(f"{n_95} components keep 95% of the information (from {X_train_small.shape[1]} features)")

In [ ]:
# Show the data in 2D using the first 2 components
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_train_small)

plt.figure(figsize=(8, 6))
points = plt.scatter(pca_2d[:, 0], pca_2d[:, 1], c=y_train_small.values, cmap="viridis", alpha=0.4, s=10)
plt.colorbar(points, label=TARGET)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Training sample in 2D (PCA)")
plt.show()

## D2. Factor Analysis

Factor Analysis looks for a few hidden **factors** that explain the features. The *loadings* show how strongly each feature is linked to each factor.

In [ ]:
fa = FactorAnalysis(n_components=3, random_state=RANDOM_STATE)
fa.fit(X_train_small)

loadings = pd.DataFrame(fa.components_.T, index=X_train_small.columns, columns=["Factor 1", "Factor 2", "Factor 3"])

plt.figure(figsize=(6, max(4, len(loadings) * 0.4)))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Factor loadings")
plt.show()

## D3. Cluster Analysis (K-Means)

K-Means puts similar rows into groups (clusters). We try K = 2 ... 10 and use the **elbow chart** and the **silhouette score** (higher = better) to choose K.

In [ ]:
k_values = range(2, 11)
inertias = []
silhouettes = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans.fit_predict(X_train_small)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_train_small, labels, sample_size=3000, random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow method (inertia)")
axes[0].set_xlabel("K")
axes[1].plot(list(k_values), silhouettes, marker="o", color="green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("K")
plt.show()

best_k = list(k_values)[int(np.argmax(silhouettes))]
print("Best K by silhouette score:", best_k)

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans.fit_predict(X_train_small)

print("Number of rows in each cluster:")
print(pd.Series(cluster_labels).value_counts().sort_index())
print("\nAverage target in each cluster:")
print(y_train_small.groupby(cluster_labels).mean().round(2))

plt.figure(figsize=(8, 6))
sns.scatterplot(x=pca_2d[:, 0], y=pca_2d[:, 1], hue=cluster_labels, palette="tab10", alpha=0.5, s=10)
plt.title(f"K-Means clusters (K={best_k}) shown in 2D PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# E. Machine Learning Models

**Data splitting (80 / 15 / 5, random_state = 123)** was already done in step **C3**, before scaling, to avoid data leakage. Models are compared and tuned on the **20,000-row training sample** (C6) and always evaluated on the **full test data**.

## E1. Define the models (20 regression models)

In [ ]:
# Some models (SVR, Linear SVR, SGD, MLP) work badly when the target has big numbers (like 200,000).
# TransformedTargetRegressor scales the target for them and changes the predictions back automatically.
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=10000),
    "Elastic Net": ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
    "Bayesian Ridge": BayesianRidge(),
    "ARD Regression": ARDRegression(),
    "Huber Regressor": HuberRegressor(max_iter=1000),
    "SGD Regressor": TransformedTargetRegressor(regressor=SGDRegressor(random_state=RANDOM_STATE), transformer=StandardScaler()),
    "KNN": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "Extra Trees": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    "Bagging": BaggingRegressor(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostRegressor(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    "SVR": TransformedTargetRegressor(regressor=SVR(), transformer=StandardScaler()),
    "Linear SVR": TransformedTargetRegressor(regressor=LinearSVR(random_state=RANDOM_STATE, max_iter=10000), transformer=StandardScaler()),
    "MLP (Neural Network)": TransformedTargetRegressor(regressor=MLPRegressor(max_iter=1000, random_state=RANDOM_STATE), transformer=StandardScaler()),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE),
}
print("Number of models:", len(models))

## E2. Evaluation function

All the common regression metrics:
- **MAE** - average absolute error (same unit as the target)
- **MSE** - average squared error (punishes big errors)
- **RMSE** - square root of MSE (same unit as the target)
- **R2** - % of the target variation explained by the model (1 = perfect, 0 = same as predicting the mean)
- **Adjusted R2** - R2 corrected for the number of features
- **MAPE** - average % error (not good when the target has zeros)
- **Median AE** - median absolute error (not affected by a few big errors)
- **Explained Variance** - similar to R2
- **Max Error** - the worst single error

Plus **actual vs predicted** and **residual** charts for the best model.

In [ ]:
def evaluate_regression(model, X, y):
    y_pred = model.predict(X)
    n = len(y)            # number of rows
    k = X.shape[1]        # number of features
    r2 = r2_score(y, y_pred)
    adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - k - 1)
    mse = mean_squared_error(y, y_pred)
    return {
        "MAE": mean_absolute_error(y, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2,
        "Adjusted R2": adjusted_r2,
        "MAPE": mean_absolute_percentage_error(y, y_pred),
        "Median AE": median_absolute_error(y, y_pred),
        "Explained Variance": explained_variance_score(y, y_pred),
        "Max Error": max_error(y, y_pred),
    }

## E3. Cross validation (5 folds, on the training sample)

The training sample is split into 5 parts. The model trains on 4 parts and is tested on the 5th, 5 times. The average score is more reliable than one single split.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = []
for name, model in models.items():
    scores = cross_validate(model, X_train_small, y_train_small, cv=cv,
                            scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"], n_jobs=-1)
    cv_results.append({
        "Model": name,
        "CV R2": scores["test_r2"].mean(),
        "CV R2 Std": scores["test_r2"].std(),
        "CV RMSE": -scores["test_neg_root_mean_squared_error"].mean(),
        "CV MAE": -scores["test_neg_mean_absolute_error"].mean(),
    })
    print(f"{name:25s} R2 = {scores['test_r2'].mean():.4f} (+/- {scores['test_r2'].std():.4f})")

cv_results = pd.DataFrame(cv_results).set_index("Model").sort_values("CV R2", ascending=False)
cv_results

## E4. Train the models - training and testing evaluation (default parameters)

In [ ]:
train_results = {}
test_results = {}

for name, model in models.items():
    model.fit(X_train_small, y_train_small)
    train_results[name] = evaluate_regression(model, X_train_small, y_train_small)
    test_results[name] = evaluate_regression(model, X_test_sel, y_test)
    print(f"Trained: {name}")

train_results = pd.DataFrame(train_results).T
test_results = pd.DataFrame(test_results).T

### Training evaluation (on the training sample)

In [ ]:
train_results.sort_values("R2", ascending=False)

### Testing evaluation

In [ ]:
test_results.sort_values("R2", ascending=False)

## E5. Hyperparameter tuning dictionary

For each model we write the parameters and the values to try. (Small grids so it does not take too long - you can add more values.)

In [ ]:
# For models inside TransformedTargetRegressor the parameter name starts with "regressor__"
param_grids = {
    "Linear Regression": {"fit_intercept": [True, False]},
    "Ridge": {"alpha": [0.1, 1, 10, 100]},
    "Lasso": {"alpha": [0.01, 0.1, 1, 10]},
    "Elastic Net": {"alpha": [0.01, 0.1, 1], "l1_ratio": [0.2, 0.5, 0.8]},
    "Bayesian Ridge": {"alpha_1": [1e-6, 1e-4], "lambda_1": [1e-6, 1e-4]},
    "ARD Regression": {"alpha_1": [1e-6, 1e-4], "lambda_1": [1e-6, 1e-4]},
    "Huber Regressor": {"epsilon": [1.35, 1.5, 2.0], "alpha": [0.0001, 0.01]},
    "SGD Regressor": {"regressor__alpha": [0.0001, 0.001, 0.01], "regressor__penalty": ["l2", "l1"]},
    "KNN": {"n_neighbors": [3, 5, 7, 11], "weights": ["uniform", "distance"]},
    "Decision Tree": {"max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 5, 10]},
    "Random Forest": {"n_estimators": [100, 200], "max_depth": [5, 10, None]},
    "Extra Trees": {"n_estimators": [100, 200], "max_depth": [5, 10, None]},
    "Bagging": {"n_estimators": [10, 50], "max_samples": [0.5, 1.0]},
    "AdaBoost": {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1, 1.0]},
    "Gradient Boosting": {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
    "Hist Gradient Boosting": {"learning_rate": [0.05, 0.1], "max_depth": [None, 5], "max_iter": [100, 200]},
    "SVR": {"regressor__C": [1, 10], "regressor__epsilon": [0.1, 0.2]},
    "Linear SVR": {"regressor__C": [0.1, 1, 10]},
    "MLP (Neural Network)": {"regressor__hidden_layer_sizes": [(50,), (100,), (50, 25)], "regressor__alpha": [0.0001, 0.001]},
    "XGBoost": {"n_estimators": [100, 200], "max_depth": [3, 5], "learning_rate": [0.05, 0.1]},
}

for name, grid in param_grids.items():
    print(f"{name}: {grid}")

## E6. Tune the models with Grid Search (GridSearchCV)

Grid search tries **every combination** of the values with 3-fold cross validation and keeps the best one (by R2). This cell can take several minutes.

In [ ]:
tuned_models = {}
best_params = {}

for name, model in models.items():
    grid = GridSearchCV(model, param_grids[name], cv=3, scoring="r2", n_jobs=-1)
    grid.fit(X_train_small, y_train_small)
    tuned_models[name] = grid.best_estimator_
    best_params[name] = grid.best_params_
    print(f"{name:25s} best CV R2 = {grid.best_score_:.4f}  params = {grid.best_params_}")

In [ ]:
tuned_train_results = {}
tuned_test_results = {}
for name, model in tuned_models.items():
    tuned_train_results[name] = evaluate_regression(model, X_train_small, y_train_small)
    tuned_test_results[name] = evaluate_regression(model, X_test_sel, y_test)

tuned_train_results = pd.DataFrame(tuned_train_results).T
tuned_test_results = pd.DataFrame(tuned_test_results).T

### Tuned models - training evaluation

In [ ]:
tuned_train_results.sort_values("R2", ascending=False)

### Tuned models - testing evaluation

In [ ]:
tuned_test_results.sort_values("R2", ascending=False)

## E7. Compare the models and select the best

We choose the model with the **highest R2 on the test data** after tuning. We also check the **gap** between train and test R2: a big gap means **overfitting** (the model memorised the training data).

In [ ]:
comparison = pd.DataFrame({
    "CV R2 (default)": cv_results["CV R2"],
    "Test R2 (default)": test_results["R2"],
    "Train R2 (tuned)": tuned_train_results["R2"],
    "Test R2 (tuned)": tuned_test_results["R2"],
    "Test RMSE (tuned)": tuned_test_results["RMSE"],
    "Test MAE (tuned)": tuned_test_results["MAE"],
})
comparison["Overfit gap (train - test)"] = comparison["Train R2 (tuned)"] - comparison["Test R2 (tuned)"]
comparison = comparison.sort_values("Test R2 (tuned)", ascending=False)
comparison

In [ ]:
comparison[["Test R2 (default)", "Test R2 (tuned)"]].sort_values("Test R2 (tuned)").plot(kind="barh", figsize=(10, 9))
plt.title("Test R2 - default vs tuned")
plt.xlabel("R2")
plt.xlim(left=max(-1, comparison[["Test R2 (default)", "Test R2 (tuned)"]].min().min() - 0.05))
plt.show()

best_model_name = comparison.index[0]
print("Best model:", best_model_name)
print("Best parameters:", best_params[best_model_name])

### Re-train the best model on the full training data

Until now the models were trained on the 20,000-row sample. Now we train the best model (with its best parameters) on **all** the training rows - more data usually gives a better model. (`clone` makes a new, untrained copy with the same parameters.)

Exception: **SVM** (`SVC` / `SVR`) becomes very slow with many rows (the training time grows about with rows x rows), so if the best model is an SVM we keep the one trained on the sample.

In [ ]:
if best_model_name in ["SVC", "SVR"]:
    best_model = tuned_models[best_model_name]
    print("SVM: we keep the model trained on the sample (re-training on all rows would take a very long time)")
else:
    best_model = clone(tuned_models[best_model_name])
    best_model.fit(X_train_sel, y_train)

best_test_scores = evaluate_regression(best_model, X_test_sel, y_test)
pd.DataFrame({
    "Trained on the sample": tuned_test_results.loc[best_model_name],
    "Trained on full training data": pd.Series(best_test_scores),
})

In [ ]:
# Details of the best model on the test data
y_test_pred = best_model.predict(X_test_sel)
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(y_test, y_test_pred, alpha=0.2, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title("Actual vs Predicted (test)")
axes[1].scatter(y_test_pred, residuals, alpha=0.2, s=8)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Residual plot (test)")
sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].set_title("Residual distribution (test)")
plt.tight_layout()
plt.show()

## E8. Save the best model

We save the model **and** the scaler and the list of selected features, because new data must be prepared in the same way before predicting.

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(best_model, "models/best_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(selected_features, "models/selected_features.pkl")

print(f"Saved: {best_model_name} -> models/best_model.pkl")

## E9. Import (load) the saved model

In [ ]:
loaded_model = joblib.load("models/best_model.pkl")
loaded_scaler = joblib.load("models/scaler.pkl")
loaded_features = joblib.load("models/selected_features.pkl")
print(loaded_model)

# Prepare the unseen data exactly like the training data: scale -> keep the selected features
X_unseen_loaded = pd.DataFrame(loaded_scaler.transform(X_unseen), columns=X.columns, index=X_unseen.index)[loaded_features]
X_unseen_loaded.head()

## E10. Predict on the 5% unseen data and evaluate

This data was never used for training, tuning or choosing the model - it is the most honest check.

In [ ]:
y_unseen_pred = loaded_model.predict(X_unseen_loaded)
unseen_scores = evaluate_regression(loaded_model, X_unseen_loaded, y_unseen)

pd.DataFrame({
    "Test (15%)": pd.Series(best_test_scores),
    "Unseen (5%)": pd.Series(unseen_scores),
})

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_unseen, y_unseen_pred, alpha=0.3, s=8, color="green")
plt.plot([y_unseen.min(), y_unseen.max()], [y_unseen.min(), y_unseen.max()], color="red", linestyle="--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted (unseen 5%)")
plt.show()

In [ ]:
# Actual vs predicted (first 20 rows)
result = pd.DataFrame({"Actual": y_unseen.values, "Predicted": y_unseen_pred.round(2)}, index=y_unseen.index)
result["Error"] = (result["Actual"] - result["Predicted"]).round(2)
result.head(20)

# Summary

1. We cleaned a messy dataset: dropped IDs, renamed columns, converted numbers saved as text, fixed messy text and short codes,
   converted dates, handled missing values (drop column > 50%, mean / mode filling), fixed wrong values and capped outliers,
   removed duplicate rows / columns and one-value columns.
2. We explored the target, the correlations and every feature with charts.
3. We created new features (binning), encoded the text columns, split the data **80 / 15 / 5**, scaled it and selected the best features.
4. We used PCA, Factor Analysis and K-Means to understand the structure of the data.
5. We trained **20 models**, used **cross validation**, evaluated them on train and test, tuned them with **Grid Search**, and chose the best by test **R2**.
6. We re-trained the best model on the full training data, saved it, loaded it again and checked it on the **5% unseen data**.

**Ideas to try next:** a bigger training sample (`SAMPLE_SIZE`), more values in the grids, a different number of features `K`, log-transforming the target (`np.log1p`) and changing it back with `np.expm1`, or adding the K-Means cluster as a new feature.